# Hybrid GNN-LSTM static replication (advanced)

This tutorial documents **end-to-end** how to run the **Hybrid GNN-LSTM static replication model**: predict **exotic target trade P&L (hence price)** from (i) **elementary trade P&L time series**, (ii) an **adjacency matrix** over elementary and target trades, and (iii) **encoded trade attribute features**. We use the library's data builders, encoder, and model so the pipeline is efficient, modular, and runs cleanly.

## What you'll learn

| Section | Topics |
|---------|--------|
| **Objective & problem** | Static replication: exotic P&L from elementary P&L and trade structure |
| **Mathematics** | Full formulation: GNN, RNN, fusion, readout, loss (PhD-level) |
| **Model inputs** | The three inputs (PnL series, adjacency, features) and where they come from |
| **Data** | Build dataset (FX or synthetic), train/val/projection split |
| **Model & training** | HybridGnnRnn config, dataset, fit; minimal notebook code |
| **Evaluation** | MSE, MAE, R², residuals on validation and projection |
| **Visualisations** | Graph, architecture, training curves, predicted vs actual, interpretation |
| **Summary & next steps** | Recap and future improvements (GPU, path-wise PnL) |

---
## 0. Configuration (edit and run)

Data, split, and training parameters only. **Model parameters** (architecture, units, layers, etc.) are in the **standalone model config cell** in Section 4—that is the only place to change the hybrid model definition. **Data path:** `USE_SYNTHETIC_DATA = True` uses fully synthetic features and PnL (fast); `False` uses real FX instruments and pricers for features, with synthetic PnL for speed. See **Section 3.2.1** for a detailed description of each path.

**Running this notebook:** Run cells top to bottom. For a quicker run, set `USE_SYNTHETIC_DATA = True` and optionally reduce `N_SAMPLES` (e.g. 300) and `EPOCHS` (e.g. 5). For full FX portfolio data and training, keep `USE_SYNTHETIC_DATA = False` and run all cells (data build may take a few minutes).

In [ ]:
# ============== Data ==============
USE_SYNTHETIC_DATA = False  # True: fast synthetic data; False: FX portfolio (instruments + encoder)
N_ELEMENTARY = 200          # Elementary trades (vanilla + digital). Use 1000 for full scale.
N_TARGETS = 20             # Exotic target trades (barrier + double barrier + asian + touch)
N_BARRIER = 5
N_DOUBLE_BARRIER = 5
N_ASIAN = 5
N_TOUCH = 5
N_SAMPLES = 600            # Number of scenarios (samples)
N_TIMESTEPS = 24           # PnL history length per sample
K_NEIGHBOURS = 10          # k-NN graph
SPOT = 1.10
SIGMA = 0.15
NOISE_STD = 0.5

# ============== Train / Val / Projection ==============
TRAIN_RATIO = 0.6
VAL_RATIO = 0.2
PROJECTION_RATIO = 0.2

# ============== Training ==============
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
PATIENCE_EARLY = 10
PATIENCE_LR = 5

SEED = 42

In [ ]:
# Hybrid GNN-LSTM model config (model parameters only). Edit and re-run; next cell builds the model.
model_config = {
    "general": {
        "architecture": "default",
    },
    "gnn_model": {
        "general": {
            "layers": 2,
            "layer_type": "mixed_graph_sage",
            "dropout_rate": 0.1,
            "use_bias": True,
            "use_residual": True,
            "layer_norm": True,
        },
        "parameters": {
            "units": 64,
            "activation": "relu",
            "kernel_initializer": "glorot_uniform",
            "bias_initializer": "zeros",
        },
    },
    "rnn_model": {
        "general": {
            "layers": 3,
            "layer_type": "lstm",
            "dropout_rate": 0.1,
        },
        "parameters": {
            "units": 128,
            "activation": "tanh",
            "recurrent_activation": "sigmoid",
            "kernel_initializer": "glorot_uniform",
            "recurrent_initializer": "orthogonal",
            "bias_initializer": "zeros",
        },
    },
    "fusion_model": {
        "general": {
            "dropout_rate": 0.1,
            "fusion_mode": "gate",
            "num_heads": 1,
        },
        "parameters": {
            "units": 64,
            "activation": None,
            "kernel_initializer": "glorot_uniform",
            "bias_initializer": "zeros",
        },
    },
    "attention_model": {
        "general": {
            "dropout_rate": 0.0,
            "num_heads": 1,
        },
        "parameters": {
            "units": 32,
            "activation": "gelu",
            "kernel_initializer": "glorot_uniform",
            "bias_initializer": "zeros",
        },
    },
    "projection_model": {
        "general": {
            "baseline_new_mode": "output_mix",
            "use_baseline_norm": True,
            "use_attn_scale": True,
            "use_attn_bias": True,
            "dropout_rate": 0.1,
            "knn_k": 4,
            "knn_mode": "cosine_softmax",
            "knn_temperature": 5.0,
            "knn_power": 2.0,
            "knn_eps": 1e-8,
            "baseline_trade_count": N_TARGETS,
        },
        "parameters": {
            "units": 16,
            "activation": "gelu",
            "kernel_initializer": "glorot_uniform",
            "bias_initializer": "zeros",
        },
    },
}
print("Model config loaded (edit this cell to change architecture).")

In [ ]:
# Setup and imports — run after config
import sys
from pathlib import Path

def _find_project_root():
    path = Path.cwd()
    for _ in range(6):
        if (path / "src" / "m_learning").exists():
            return path
        if path.parent == path:
            break
        path = path.parent
    return Path.cwd().parents[2] if len(Path.cwd().parts) >= 3 else path

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"Project root: {project_root}")
print(f"TensorFlow: {tf.__version__}")

---
## 1. Objective and problem statement

**Goal: static replication.** Predict **exotic target trade P&L** (and hence price) from: (1) **elementary trade P&L time series**, (2) an **adjacency matrix** over both elementary and target trades, and (3) **encoded trade attribute features** for all trades. The model learns a mapping from these three inputs to target P&L so we can replicate or hedge exotics using liquid elementary trades and structure.

**Why it matters.** Exotic options (barriers, Asians, touch, etc.) are fewer, less liquid, and harder to hedge; their P&L is driven by the same risk factors as vanillas and digitals (spot, vol, rates). By observing elementary P&L over time and trade similarity (graph + features), we exploit **relational structure** and **temporal dynamics** to infer target P&L—well-suited when target trades are complex or illiquid and we have rich elementary P&L history.

---
## 2. Mathematical formulation (PhD-level)

**Sets and dimensions.** Let \\(T\\) be the set of all trades: elementary \\(\\mathcal{E}\\) with \\(|\\mathcal{E}| = E\\) and target \\(\\mathcal{N}\\) with \\(|\\mathcal{N}| = N\\), so \\(T = E + N\\). Batch size \\(B\\), number of timesteps \\(S\\), feature dimension \\(F\\). The model maps three inputs to target P&L predictions.

**Inputs (aligned with Section 3):**

- \\(\\mathbf{X} \\in \\mathbb{R}^{T \\times F}\\) — encoded trade features (elementary + target): moneyness, time-to-maturity, delta, vega, product embeddings from `TradeAttributeEncoder`.
- \\(\\mathbf{A} \\in \\mathbb{R}^{T \\times T}\\) — row-normalised adjacency over all trades (e.g. k-NN from \\(\\mathbf{X}\\)); same graph for every sample in a batch.
- \\(\\mathbf{P} \\in \\mathbb{R}^{B \\times S \\times E}\\) — elementary P&L time series: for each of \\(B\\) samples, \\(S\\) timesteps, \\(E\\) elementary trades.

**GNN block.** Message passing over the graph and features: \\(\\mathbf{H}^{(0)} = \\mathbf{X}\\), and for layer \\(\\ell\\):
\\[
  \\mathbf{H}^{(\\ell+1)} = \\sigma\\bigl( \\mathbf{A} \\mathbf{H}^{(\\ell)} \\mathbf{W}^{(\\ell)} \\bigr),
\\]
with non-linearity \\(\\sigma\\) and learned \\(\\mathbf{W}^{(\\ell)}\\). Output \\(\\mathbf{H}^{\\mathrm{gnn}} \\in \\mathbb{R}^{T \\times d_g}\\) (one embedding per trade). This captures **structural** similarity (who is close to whom in feature space).

**RNN block.** LSTM over the time axis of \\(\\mathbf{P}\\): at each step the input is the vector of elementary P&L; the final hidden state (or pooled output) gives a single embedding \\(\\mathbf{h}^{\\mathrm{rnn}} \\in \\mathbb{R}^{d_r}\\) per sample. This captures **temporal** patterns in elementary P&L.

**Fusion.** Combine GNN embeddings \\(\\mathbf{H}^{\\mathrm{gnn}}\\) with the RNN embedding \\(\\mathbf{h}^{\\mathrm{rnn}}\\) (repeated/broadcast over trades) via gating or attention to produce fused features \\(\\mathbf{Z} \\in \\mathbb{R}^{T \\times d_f}\\) per sample (so \\(\\mathbf{Z}\\) has shape \\(B \\times T \\times d_f\\) in batch form).

**Target readout.** For each target index \\(n \\in \\mathcal{N}\\), the model attends over \\(\\mathbf{Z}\\) (and optionally \\(\\mathbf{A}\\)) and projects to a scalar \\(\\hat{y}_{b,n}\\). Output \\(\\hat{\\mathbf{Y}} \\in \\mathbb{R}^{B \\times N}\\).

**Loss.** Mean squared error over batch and targets:
\\[
  \\mathcal{L} = \\frac{1}{BN} \\sum_{b=1}^{B} \\sum_{n \\in \\mathcal{N}} (y_{b,n} - \\hat{y}_{b,n})^2.
\\]

Implementation: see `HybridGnnRnn.run_default_model` in `src.m_learning.models.gnn_rnn_hybrid.hybrid_model` (GNN → RNN → fusion → target attention → projection).

### 3.2.1 Synthetic vs FX: what each path entails

The config flag **`USE_SYNTHETIC_DATA`** in Section 0 chooses between two data paths. Both produce the same three inputs (trade features, adjacency, PnL history) and targets; only how they are built differs.

---

**`USE_SYNTHETIC_DATA = True` (synthetic path)**

- **Purpose:** Fast, self-contained run with no market or pricers. Good for testing the model and pipeline.
- **Trade features:** Generated at random: moneyness, time-to-maturity, delta, vega, and a one-hot product type (3 classes). No real instruments; features are drawn from fixed ranges (e.g. moneyness in [0.8, 1.2]).
- **Elementary vs target:** The first E trades are treated as elementary, the next N as targets (or a random disjoint split). There is no product-type distinction (no vanilla vs barrier); it is purely positional.
- **Adjacency:** k-NN on the synthetic feature matrix (Euclidean distance), then row-normalised. Same graph for all samples.
- **Elementary PnL history:** Random walk: at each timestep, independent Gaussian increments per elementary trade, then cumulated over time. No link to real market paths.
- **Targets:** Synthetic relationship: targets = (random weighted sum of final elementary PnL) + Gaussian noise. The weights are random and fixed per run. No real pricing or risk factor; the model learns this synthetic mapping.
- **When to use:** Quick runs, CI, or when you want to try architecture/hyperparameters without building FX portfolios or calling pricers.

---

**`USE_SYNTHETIC_DATA = False` (FX portfolio path)**

- **Purpose:** Realistic trade universe and features using the library's FX instruments and pricers; PnL/targets in this tutorial are still synthetic for speed, but the graph and features reflect real option structure.
- **Key design:** Elementary and target portfolios are built **separately** using the library's `Portfolio` component. Each portfolio is a collection of `Position` objects (each holding an instrument instance). This modular design allows elementary and target trade sets to be defined, extended, and priced independently.
- **Market:** A minimal FX market is built (e.g. EURUSD spot, flat vol surface, USD and EUR discount curves) via `build_fx_market()`. Used only for pricing and greeks.
- **Elementary portfolio:** Built via `build_elementary_portfolio()`. Half vanilla European calls/puts, half digital cash options. Random strike (0.85–1.15 × spot), random expiry (0.1–2.0 years). Uses real library instruments (`FxVanillaEuropeanOption`, `FxDigitalEuropeanOption`).
- **Target portfolio:** Built via `build_target_portfolio()`. A mix of barriers, double barriers, Asians, and touch options with random parameters. Uses real library instruments (`FxBarrierEuropeanOption`, `FxDoubleBarrierEuropeanOption`, `FxAsianEuropeanOption`, `FxTouchEuropeanOption`).
- **Pricing:** Both portfolios are priced via `PortfolioPricer` with `DefaultPricerRegistry()`. For elementary (vanilla, digital), closed-form BSM pricers are used. For exotics, Monte Carlo pricers run (may take a few seconds per instrument).
- **Trade features:** From pricing results we read PV, delta, and vega. Combined with strike and expiry we compute moneyness (spot/strike), TTM, and one-hot product subtype. These are stacked into the encoded feature matrix \\(\\mathbf{X}\\).
- **Adjacency:** k-NN on \\(\\mathbf{X}\\) (Euclidean), then row-normalised. Trades similar in moneyness, maturity, delta, vega, and product type sit close in the graph.
- **PnL and targets:** This tutorial still uses synthetic PnL and targets (random walk + weighted sum) so the notebook runs quickly. In production, replace with path-wise repricing.
- **When to use:** When you want a realistic trade graph and feature set (real instruments, real greeks) and are willing to pay the cost of building the FX portfolios and running pricers once per instrument.

---
## 3. Model inputs and data processing

### 3.1 Model inputs (the three inputs to the model)

The Hybrid GNN-LSTM expects exactly three tensor inputs (plus index vectors that select target and elementary nodes):

| Input | Shape | Description | Produced by |
|-------|--------|-------------|-------------|
| **Elementary P&L time series** | \\((n\\_samples, n\\_timesteps, n\\_elementary)\\) | Observed P&L history per scenario | Data builder (synthetic here; path-wise repricing in production). Fed as `pnl_history`. |
| **Adjacency matrix** | \\((n\\_trades, n\\_trades)\\) | Graph over **all** trades (elementary + target) | k-NN on encoded features; row-normalised. Fed as `adjacency_matrix`. |
| **Encoded trade features** | \\((n\\_trades, n\\_features)\\) | One vector per trade (elementary + target) | `TradeAttributeEncoder` on raw attributes. Fed as `trade_features`. |

The model also receives `target_indices` and `elementary_indices`, which select which rows of the graph/feature matrix correspond to target vs elementary trades. The same graph and features are shared across all samples in a batch; only `pnl_history` varies per sample.

### 3.2 Data requirements and processing

**How we obtain the three inputs:** Build elementary portfolio (vanilla + digital) and target portfolio (exotics) **separately** using the library's Portfolio component; price each with PortfolioPricer; extract features (moneyness, TTM, delta, vega, product type); encode and build k-NN on \\(\\mathbf{X}\\) to get \\(\\mathbf{A}\\); generate (or load) PnL history and targets. Then split by time index into train / validation / projection. The library's `portfolio_builder` module does all of this; we call `build_fx_gnn_data()` and `train_val_projection_split()`.

In [ ]:
# Build data using the library. Elementary and target portfolios are built separately.
# Synthetic path: fast, no market/pricers. FX path: real instruments, real pricing/greeks.
if USE_SYNTHETIC_DATA:
    from src.m_learning.data.gnn_synthetic import generate_synthetic_gnn_data
    from src.m_learning.data.portfolio_builder import train_val_projection_split
    data = generate_synthetic_gnn_data(
        n_trades=N_ELEMENTARY + N_TARGETS,
        n_elementary=N_ELEMENTARY,
        n_targets=N_TARGETS,
        n_samples=N_SAMPLES,
        n_timesteps=N_TIMESTEPS,
        k_neighbours=K_NEIGHBOURS,
        noise_std=NOISE_STD,
        seed=SEED,
    )
else:
    from src.m_learning.data.portfolio_builder import (
        build_fx_gnn_data,
        train_val_projection_split,
    )
    # Elementary portfolio (vanilla + digital) and target portfolio (exotics) built separately,
    # priced, features extracted, and combined into GNN inputs.
    data = build_fx_gnn_data(
        n_vanilla=N_ELEMENTARY // 2,
        n_digital=N_ELEMENTARY - N_ELEMENTARY // 2,
        n_barrier=N_BARRIER,
        n_double_barrier=N_DOUBLE_BARRIER,
        n_asian=N_ASIAN,
        n_touch=N_TOUCH,
        n_samples=N_SAMPLES,
        n_timesteps=N_TIMESTEPS,
        k_neighbours=K_NEIGHBOURS,
        spot=SPOT,
        sigma=SIGMA,
        noise_std=NOISE_STD,
        seed=SEED,
    )

(
    train_inputs, train_targets,
    val_inputs, val_targets,
    proj_inputs, proj_targets,
) = train_val_projection_split(
    data,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    projection_ratio=PROJECTION_RATIO,
)

n_trades = data.trade_features.shape[0]
n_elem = len(data.elementary_indices)
n_tgt = len(data.target_indices)
print(f"Trades: {n_trades} (elementary: {n_elem}, targets: {n_tgt})")
print(f"Trade features: {data.trade_features.shape}")
print(f"Adjacency: {data.adjacency_matrix.shape}")
print(f"Train samples: {train_targets.shape[0]}, Val: {val_targets.shape[0]}, Projection: {proj_targets.shape[0]}")

### 3.3 Visualise trade graph (k-NN)

Nodes = trades; edges = k-NN. Below: PCA of encoded features; elementary vs target nodes. (See Section 8.1 for optional adjacency heatmap.)

In [ ]:
# Reduce to 2D for plotting (PCA of encoded features)
from sklearn.decomposition import PCA
X_2d = PCA(n_components=2, random_state=SEED).fit_transform(data.trade_features)
n_elem = len(data.elementary_indices)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(X_2d[:n_elem, 0], X_2d[:n_elem, 1], c="steelblue", s=15, alpha=0.6, label="Elementary")
ax.scatter(X_2d[n_elem:, 0], X_2d[n_elem:, 1], c="coral", s=80, marker="s", edgecolors="black", label="Targets")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Trade graph (PCA of encoded features); squares = target exotics")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 4. Model: build HybridGnnRnn

A **standalone config cell** below defines the full model parameters (general, gnn_model, rnn_model, fusion_model, attention_model, projection_model). Edit that dict and re-run it; then the next cell builds `BatchedHybridGnnRnn` from `model_config` and compiles. The wrapper handles batched `tf.data.Dataset` (tiled graph inputs) correctly. Config dimensions (e.g. units in each block) correspond to \\(d_g\\), \\(d_r\\), and \\(d_f\\) in Section 2.

### Model config (standalone)

Example configuration for the Hybrid GNN-LSTM (model parameters only). Edit this dict and re-run the cell; the next cell builds the model from it.

In [ ]:
# Hybrid GNN-LSTM model config (model parameters only). Edit and re-run; next cell builds the model.
model_config = {
    "general": {
        "architecture": "default",
    },
    "gnn_model": {
        "general": {
            "layers": 2,
            "layer_type": "mixed_graph_sage",
            "dropout_rate": 0.1,
            "use_bias": True,
            "use_residual": True,
            "layer_norm": True,
        },
        "parameters": {
            "units": 64,
            "activation": "relu",
            "kernel_initializer": "glorot_uniform",
            "bias_initializer": "zeros",
        },
    },
    "rnn_model": {
        "general": {
            "layers": 3,
            "layer_type": "lstm",
            "dropout_rate": 0.1,
        },
        "parameters": {
            "units": 128,
            "activation": "tanh",
            "recurrent_activation": "sigmoid",
            "kernel_initializer": "glorot_uniform",
            "recurrent_initializer": "orthogonal",
            "bias_initializer": "zeros",
        },
    },
    "fusion_model": {
        "general": {
            "dropout_rate": 0.1,
            "fusion_mode": "gate",
            "num_heads": 1,
        },
        "parameters": {
            "units": 64,
            "activation": None,
            "kernel_initializer": "glorot_uniform",
            "bias_initializer": "zeros",
        },
    },
    "attention_model": {
        "general": {
            "dropout_rate": 0.0,
            "num_heads": 1,
        },
        "parameters": {
            "units": 32,
            "activation": "gelu",
            "kernel_initializer": "glorot_uniform",
            "bias_initializer": "zeros",
        },
    },
    "projection_model": {
        "general": {
            "baseline_new_mode": "output_mix",
            "use_baseline_norm": True,
            "use_attn_scale": True,
            "use_attn_bias": True,
            "dropout_rate": 0.1,
            "knn_k": 4,
            "knn_mode": "cosine_softmax",
            "knn_temperature": 5.0,
            "knn_power": 2.0,
            "knn_eps": 1e-8,
            "baseline_trade_count": N_TARGETS,
        },
        "parameters": {
            "units": 16,
            "activation": "gelu",
            "kernel_initializer": "glorot_uniform",
            "bias_initializer": "zeros",
        },
    },
}
print("Model config loaded (edit this cell to change architecture).")

In [ ]:
from src.m_learning.models.gnn_rnn_hybrid.wrapper import BatchedHybridGnnRnn

# model_config is defined in the standalone cell above
model = BatchedHybridGnnRnn(model_config, name="hybrid_pnl_advanced")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mse",
    metrics=["mae"],
)
print("Model built and compiled.")

---
## 5. Training Pipeline

Convert train/val inputs and targets to `tf.data.Dataset`, then fit with early stopping and learning-rate reduction.

In [ ]:
from src.m_learning.data.portfolio import gnn_inputs_to_tf_dataset

train_ds = gnn_inputs_to_tf_dataset(train_inputs, train_targets, batch_size=BATCH_SIZE, shuffle=True)
val_ds = gnn_inputs_to_tf_dataset(val_inputs, val_targets, batch_size=BATCH_SIZE, shuffle=False)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=PATIENCE_EARLY, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=PATIENCE_LR, min_lr=1e-5, verbose=1
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)
print(f"\nFinal train loss: {history.history['loss'][-1]:.6f}")
print(f"Final val loss: {history.history['val_loss'][-1]:.6f}")

---
## 6. Training analytics

Loss and MAE curves below show how well the model fits the training set and generalises to the validation set.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history.history["loss"]) + 1)
axes[0].plot(epochs_range, history.history["loss"], label="Train loss", linewidth=2)
axes[0].plot(epochs_range, history.history["val_loss"], label="Val loss", linewidth=2)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE"); axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs_range, history.history["mae"], label="Train MAE", linewidth=2)
axes[1].plot(epochs_range, history.history["val_mae"], label="Val MAE", linewidth=2)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MAE"); axes[1].set_title("MAE"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 7. Evaluation: validation and projection

We collect predictions on the validation and projection sets (using the same batch loop for consistency), then compute MSE, MAE, R², and residual percentiles. Projection metrics measure out-of-sample predictive power.

In [ ]:
def collect_predictions(inputs_dict, targets, model):
    n = targets.shape[0]
    preds_list = []
    # Batch through (same batch_size as training for consistency)
    for start in range(0, n, BATCH_SIZE):
        end = min(start + BATCH_SIZE, n)
        batch_inputs = {k: v[start:end] for k, v in inputs_dict.items()}
        preds = model(batch_inputs, training=False)
        preds_list.append(preds.numpy())
    y_pred = np.concatenate(preds_list, axis=0).flatten()
    y_true = targets.flatten()
    return y_true, y_pred

y_true_val, y_pred_val = collect_predictions(val_inputs, val_targets, model)
y_true_proj, y_pred_proj = collect_predictions(proj_inputs, proj_targets, model)

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

for name, y_true, y_pred in [("Validation", y_true_val, y_pred_val), ("Projection", y_true_proj, y_pred_proj)]:
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    res = y_true - y_pred
    print(f"{name}: MSE={mse:.6f}, MAE={mae:.6f}, R²={r2:.4f}, P90|res|={np.percentile(np.abs(res), 90):.6f}")

---
## 8. Visualisations

### 8.1 Inputs and graph

The trade graph (PCA of encoded features) is in Section 3.3. Below: optional adjacency matrix heatmap (same graph used by the model).

In [ ]:
# Adjacency matrix heatmap (row-normalised k-NN)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(data.adjacency_matrix, cmap="Blues", aspect="auto")
ax.set_xlabel("Trade index"); ax.set_ylabel("Trade index")
ax.set_title("Adjacency matrix A (row-normalised k-NN)")
plt.colorbar(im, ax=ax, label="Weight")
plt.tight_layout(); plt.show()

### 8.2 Model architecture

High-level data flow (Section 2): **Inputs** (X, A, P) → **GNN block** → **RNN block** → **Fusion** → **Target attention** → **Projection** → **Target P&L predictions**.

```mermaid
flowchart LR
  subgraph inputs [Model inputs]
    P["Elementary PnL P"]
    A["Adjacency A"]
    X["Encoded features X"]
  end
  subgraph model [Hybrid GNN-LSTM]
    GNN[GNN block]
    RNN[RNN block]
    FUS[Fusion]
    ATT[Target attention]
    PROJ[Projection]
  end
  P --> RNN
  X --> GNN
  A --> GNN
  GNN --> FUS
  RNN --> FUS
  A --> FUS
  FUS --> ATT
  A --> ATT
  ATT --> PROJ
  PROJ --> Y["Target PnL predictions"]
```

### 8.3 Training curves

See **Section 6** for loss and MAE curves (train vs validation).

### 8.4 Predictive power

Predicted vs actual P&L (validation and projection) and residual distributions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
# Validation: predicted vs actual
ax = axes[0]
ax.scatter(y_true_val, y_pred_val, alpha=0.5, s=20)
lims = [min(y_true_val.min(), y_pred_val.min()), max(y_true_val.max(), y_pred_val.max())]
ax.plot(lims, lims, "r--", lw=2, label="45° line")
ax.set_xlabel("Actual P&L"); ax.set_ylabel("Predicted P&L")
ax.set_title(f"Validation: Predicted vs Actual (R²={r2_score(y_true_val, y_pred_val):.4f})")
ax.legend(); ax.grid(True, alpha=0.3); ax.set_xlim(lims); ax.set_ylim(lims)
# Projection: predicted vs actual
ax = axes[1]
ax.scatter(y_true_proj, y_pred_proj, alpha=0.5, s=20, color="green")
lims2 = [min(y_true_proj.min(), y_pred_proj.min()), max(y_true_proj.max(), y_pred_proj.max())]
ax.plot(lims2, lims2, "r--", lw=2, label="45° line")
ax.set_xlabel("Actual P&L"); ax.set_ylabel("Predicted P&L")
ax.set_title(f"Projection: Predicted vs Actual (R²={r2_score(y_true_proj, y_pred_proj):.4f})")
ax.legend(); ax.grid(True, alpha=0.3); ax.set_xlim(lims2); ax.set_ylim(lims2)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(y_true_val - y_pred_val, bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Residual (Actual - Predicted)"); axes[0].set_ylabel("Count")
axes[0].set_title("Validation residuals")
axes[1].hist(y_true_proj - y_pred_proj, bins=50, edgecolor="black", alpha=0.7, color="green")
axes[1].set_xlabel("Residual"); axes[1].set_ylabel("Count")
axes[1].set_title("Projection residuals")
plt.tight_layout(); plt.show()

### 8.5 Learning and interpretation

- **R² and residuals:** R² close to 1 means predictions track actual P&L well; projection R² shows out-of-sample generalisation. Residuals centred near zero with moderate spread indicate unbiased predictions; heavy tails suggest some scenarios or targets are harder to predict.
- **Generalisation:** Compare validation vs projection metrics; similar performance suggests the model learned generalisable structure rather than overfitting.
- **Optional below:** Residual vs a sample-level feature (e.g. mean final elementary PnL) to check if errors are structured by scenario.

In [ ]:
# Optional: residual vs mean final elementary PnL (per sample)
mean_final_pnl = proj_inputs["pnl_history"][:, -1, :].mean(axis=1)  # (n_proj,)
n_proj = len(mean_final_pnl)
res_proj = (y_true_proj - y_pred_proj).reshape(n_proj, -1)  # (n_proj, n_targets)
x_plot = np.repeat(mean_final_pnl, res_proj.shape[1])
y_plot = res_proj.flatten()
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_plot, y_plot, alpha=0.4, s=15)
ax.axhline(0, color="red", linestyle="--")
ax.set_xlabel("Mean final elementary P&L (per sample)"); ax.set_ylabel("Residual (actual - predicted)")
ax.set_title("Projection: residual vs scenario (mean elementary P&L)")
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
## 9. Summary and next steps

**Recap.** This tutorial ran the **Hybrid GNN-LSTM static replication** model end-to-end: we fed the three inputs (elementary P&L time series, adjacency matrix, encoded trade features), built and trained the model with the library's data builders and dataset utilities, and evaluated it on validation and projection sets. The model learns to predict exotic target P&L from structure and temporal dynamics; metrics (MSE, MAE, R², residuals) and the plots in Section 8 show how well it learned and how well it generalises.

**Next steps.** See Section 10 for future improvements (GPU training, general training pipeline, path-wise PnL, hyperparameter tuning).

---
## 10. Future Improvements

- **GPU training:** Move the model and data pipeline to GPU by ensuring all tensors are created on the default device (e.g. `tf.config.set_visible_devices`) and using `tf.data.Dataset` with `.prefetch(tf.data.AUTOTUNE)` and optional `.cache()`. The training pipeline is already Keras-based, so running under a GPU-enabled TensorFlow build will use the GPU for the hybrid model. For multi-GPU, use `tf.distribute.MirroredStrategy()` and wrap model creation and `model.fit` inside the strategy scope.

- **More general training pipeline:** Abstract the training loop (config → data → split → dataset → model → fit → evaluate) into a reusable runner (e.g. in `src.m_learning.calibration.training_manager` or a dedicated `gnn_hybrid_trainer`). That runner can accept a data builder callable, model config, and training hyperparameters, and return history and evaluation metrics. This makes it easy to sweep hyperparameters or run the same pipeline on different portfolios without duplicating notebook code.

- **Path-wise PnL:** Replace synthetic PnL generation with actual path-wise repricing: build a `MarketDataset` over scenarios and dates, snapshot a `Market` at each (time_idx, scenario_idx), price the portfolio of elementary and target trades, and stack results into `pnl_history` and `targets`. That aligns the tutorial with production risk systems.

- **Hyperparameter tuning:** Sweep over GNN/RNN units, layers, learning rate, k-NN, and train/val/projection split ratios (e.g. via a grid or a Bayesian optimiser) and track validation/projection R² to choose a robust configuration.